<!-- ===== Gradient Header ===== -->
<p style="background: linear-gradient(90deg, #1a237e, #1565c0, #0288d1, #26c6da);
         font-family: 'Montserrat', sans-serif;
         font-size: 26px;
         text-align: center;
         color: #ffffff;
         padding: 24px 42px;
         border-radius: 30px;
         border: 3px solid #0288d1;
         box-shadow: 0 10px 25px rgba(2,136,209,0.35);
         letter-spacing: 1.2px;
         word-spacing: 4px;
         font-weight: 700;
         text-shadow: 2px 2px 4px rgba(0,0,0,0.25);
         margin: 10px 0 20px;">
  🫀 📊 Complete EDA — Heart Disease Dataset Analysis
</p>

---

### 📌 Full Pipeline at a Glance
```
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🔵  PART 1 — EXPLORATORY DATA ANALYSIS
    Step 1  → Import Libraries
    Step 2  → Load & First Look
    Step 3  → Data Quality Check
    Step 4  → Target Variable Analysis
    Step 5  → Univariate Analysis
    Step 6  → Bivariate Analysis
    Step 7  → Correlation Analysis
    Step 8  → Statistical Tests
    Step 9  → Advanced Plots
    Step 10 → EDA Summary
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🟢  PART 2 — MODELING & SUBMISSION
    Step 11 → Advanced Preprocessing
    Step 12 → Feature Engineering
    Step 13 → Baseline XGBoost
    Step 14 → Hyperparameter Tuning (Optuna)
    Step 15 → Final Model Training
    Step 16 → Full Evaluation (ROC + CM + Importance)
    Step 17 → Generate Submission File
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
```

---
# 🔵 PART 1 — Exploratory Data Analysis (EDA)

## 📦 Step 1: Import Libraries

In [ ]:
# ── Core ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Visualization ─────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Statistics ────────────────────────────────────────────────────────────
from scipy import stats
from scipy.stats import mstats

# ── Preprocessing ─────────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

# ── Model ─────────────────────────────────────────────────────────────────
from xgboost import XGBClassifier

# ── Evaluation ────────────────────────────────────────────────────────────
from sklearn.metrics import (
    roc_auc_score, roc_curve, auc,
    classification_report, confusion_matrix,
    f1_score, accuracy_score
)

# ── Hyperparameter Tuning ─────────────────────────────────────────────────
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)  # Silence spam output

# ── Settings ──────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (10, 6)
# import plotly.io as pio
# pio.renderers.default = 'iframe'   # ✅ Fix for Kaggle plotly rendering
RANDOM_STATE = 42

print('✅ All libraries loaded successfully!')

## 📂 Step 2: Load & First Look at Data

In [ ]:
# Load the dataset
# ⚠️ Keep a raw copy of train/test — we'll use them later in modeling
df       = pd.read_csv('data/train.csv')
test_raw = pd.read_csv('data/test.csv')
# sub      = pd.read_csv('/kaggle/input/playground-series-s6e2/sample_submission.csv')

# Save raw train copy before any modifications
train_raw = df.copy()

print(f'📊 Train shape : {df.shape}')
print(f'📊 Test shape  : {test_raw.shape}')
# print(f'📊 Sub shape   : {sub.shape}')
# print(f'\n📋 Sample submission format (what Kaggle expects from us):')
# print(sub.head(3))
print('\n🔍 First 5 rows of train:')
df.head()

In [ ]:
# Column info - what data types do we have?
print('📌 Column Names and Data Types:')
print(df.dtypes)
print('\n📌 Detailed Info:')
df.info()

In [ ]:
# Basic statistics for numeric columns
print('📊 Statistical Summary:')
df.describe().round(2)

## 🫀 Feature Dictionary — What Every Column Really Means
> 📖 No medical degree needed — each feature explained in plain English with ML importance rating

---

### 🔢 `id` — Row Identifier
> ❌ **DROP — Zero ML Value**

Patient registration number. Uniquely identifies a row but tells the model **nothing** about disease. Always drop before training.

---

### 🎂 `Age` — Patient Age (29–77 years)
> ✅ **Moderate Predictor | corr: 0.21**

Arteries stiffen and accumulate plaque with age. Risk jumps significantly after **55–60**. Think of it like a car engine — old engine = worn parts = higher failure risk. Risk threshold at ~55 is where disease cases overtake healthy ones.

---

### 🚻 `Sex` — 0=Female, 1=Male
> ✅ **Strong Predictor | corr: 0.34**

Males have **~55% disease rate** vs females at **~17%** — nearly 3× higher risk. Women are protected by estrogen until menopause. Already encoded as 0/1, no further preprocessing needed.

---

### 💢 `Chest pain type` — 4 Categories
> 🔥 **Very Strong | corr: 0.46**

**1**=Typical Angina | **2**=Atypical | **3**=Non-cardiac | **4**=Asymptomatic

⚠️ **Counterintuitive:** Type 4 (no pain at all) has the **highest disease rate ~70%**. Heart disease is a silent killer — no pain doesn't mean no disease.

---

### 🩸 `BP` — Systolic Blood Pressure (94–200 mmHg)
> ⚠️ **Surprisingly Weak | corr: −0.01**

Peak pressure when heart pumps blood. Medically famous risk factor — but **near-zero ML correlation** as a standalone feature. Key lesson: **medical importance ≠ ML importance**. Keep for ensemble models; don't over-rely on it alone.

---

### 🧈 `Cholesterol` — Blood Fat Level (126–564 mg/dL)
> ⚠️ **Weak Standalone | corr: 0.08**

LDL ("bad") cholesterol sticks to artery walls → builds plaque → narrowing → heart disease. Has upper outliers up to 564 — apply **Winsorization**. Weak alone but gains power in feature combinations.

---

### 🍬 `FBS over 120` — Fasting Blood Sugar Flag
> ⚠️ **Weakest Feature | corr: 0.03**

**0** = sugar ≤ 120 mg/dL | **1** = sugar > 120 mg/dL (diabetic signal). 92% of patients = 0, making it heavily imbalanced. Diabetes links to heart disease, but this binary flag alone carries very little predictive signal.

---

### 📈 `EKG results` — Heart Electrical Signal
> ✅ **Moderate | corr: 0.22**

**0**=Normal | **1**=ST-T Abnormality (only 1,322 cases — nearly noise!) | **2**=LV Hypertrophy (enlarged heart chamber)

⚠️ **Merge or drop category 1** — 0.2% of data confuses models more than it helps.

---

### 💓 `Max HR` — Peak Heart Rate During Stress Test (71–202 bpm)
> 🔥 **Strong NEGATIVE | corr: −0.44**

The **only negative predictor** — higher Max HR = healthier heart. Formula: `Expected Max HR = 220 − Age`. Patients who couldn't exceed 100 bpm during exercise are almost all disease cases. **Never drop a feature because its correlation is negative** — magnitude is what matters.

---

### 🏃 `Exercise angina` — Chest Pain During Exercise
> 🔥 **Very Strong | corr: 0.44**

**0**=No | **1**=Yes. No exercise pain → ~30% disease rate. Yes → **~80% disease rate**. Blockages invisible at rest reveal themselves when the heart demands more blood during exercise.

---

### 📉 `ST depression` — EKG Dip During Exercise (0–6.2 mm)
> 🔥 **Strong | corr: 0.43**

How much the EKG signal dips during exercise vs rest. Bigger dip = worse blockage. Absence median ≈ 0, Presence median ≈ 1.5. **Heavily right-skewed** — apply `np.log1p()` transformation before modeling.

---

### 📐 `Slope of ST` — EKG Signal Direction at Peak Exercise
> 🔥 **Strong | corr: 0.42**

**1**=Upsloping ✅ | **2**=Flat ⚠️ | **3**=Downsloping 🔴

Works hand-in-hand with ST depression. Downsloping at peak exercise = one of the most reliable disease markers in cardiology.

---

### 🔬 `Number of vessels fluro` — Blocked Arteries Count (0–3)
> 🔥 **Very Strong | corr: 0.44**

Live X-ray counts physically blocked arteries: **0→30%**, **1→70%**, **2→88%**, **3→90%+** disease rate. A perfect monotonic relationship — every additional blocked artery increases risk. Most direct physical measurement of disease severity.

---

### ☢️ `Thallium` — Nuclear Blood Flow Imaging
> 👑 **#1 Predictor | corr: 0.61**

**3**=Normal ✅ | **6**=Fixed Defect (old heart attack scar) 🔴 | **7**=Reversible Defect (blockage under stress) 🔴

The **single strongest predictor**. Directly photographs blood flow inside heart muscle. Reversible defect = ~70% disease rate. Normal = ~20%.

---

### 🎯 `Heart Disease` — TARGET VARIABLE
> 🏷️ **PREDICT THIS**

Final diagnosis: **"Presence"** (44.8% → 282,454 patients) or **"Absence"** (55.2% → 347,546 patients). Nearly balanced — no SMOTE needed.
```python
df['Heart Disease'] = df['Heart Disease'].map({'Presence': 1, 'Absence': 0})
```

## 🔍 Step 3: Data Quality Check
> **Why?** Before analysis, always check: missing values, duplicates, and data types

In [ ]:
# ---- CHECK 1: Missing Values ----
print('='*45)
print('🔴 MISSING VALUES CHECK')
print('='*45)
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct.round(2)})
print(missing_df)
print(f'\n✅ Total missing values: {missing.sum()}')

In [ ]:
# ---- CHECK 2: Duplicates ----
print('='*45)
print('🔴 DUPLICATES CHECK')
print('='*45)
dup_count = df.duplicated().sum()
print(f'Duplicate rows: {dup_count}')
print('✅ No duplicates!' if dup_count == 0 else f'⚠️ Found {dup_count} duplicate rows!')

In [ ]:
# Drop id for EDA (raw copies still have it for modeling)
df.drop(columns='id', inplace=True)

# ---- CHECK 3: Unique Values per Column ----
print('='*45)
print('📊 UNIQUE VALUES PER COLUMN')
print('='*45)
unique_vals = df.nunique().reset_index()
unique_vals.columns = ['Column', 'Unique Values']

fig = px.bar(unique_vals, x='Column', y='Unique Values',
             title='Unique Values Count per Column',
             color='Unique Values', color_continuous_scale='Blues',
             text='Unique Values')
fig.update_layout(xaxis_tickangle=-30)
fig.show()

print(unique_vals.to_string(index=False))

📊 Unique Values — Key Takeaways

1. **Cholesterol (150), Max HR (93), BP & ST depression (66)** are true continuous features — scale them before modeling.

2. **Sex, FBS, Exercise angina (2 values), EKG, Slope, Thallium (3 values)** are categoricals hiding as integers — consider One-Hot Encoding for linear models.

3. **Simple rule from this chart:** ≤ 4 unique values = treat as categorical | 40+ unique values = treat as numerical. Your entire preprocessing strategy in one glance.

## 🎯 Step 4: Target Variable Analysis
> **Target column:** `Heart Disease` (Presence vs Absence)
> This is what our ML model will predict — always analyze it first!

In [ ]:
# Target Distribution
target_counts = df['Heart Disease'].value_counts()
target_pct    = df['Heart Disease'].value_counts(normalize=True) * 100

print('🎯 Target Variable Distribution:')
print(f'  Presence : {target_counts["Presence"]:,} ({target_pct["Presence"]:.1f}%)')
print(f'  Absence  : {target_counts["Absence"]:,} ({target_pct["Absence"]:.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#E74C3C', '#2ECC71']

axes[0].bar(target_counts.index, target_counts.values, color=colors, edgecolor='black', linewidth=0.8)
axes[0].set_title('Heart Disease Count', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Heart Disease Status')
axes[0].set_ylabel('Count')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 1000, f'{v:,}', ha='center', fontweight='bold')

axes[1].pie(target_counts.values, labels=target_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, explode=(0.05, 0))
axes[1].set_title('Heart Disease Distribution (%)', fontsize=14, fontweight='bold')

plt.suptitle('🎯 Target Variable Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('target_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Insight: Check if dataset is balanced or imbalanced!')

🎯 Target Variable — Key Takeaways

1. **Nearly balanced** — 55.2% Absence vs 44.8% Presence. No SMOTE or resampling needed.

2. **Accuracy is valid here** — but still prefer **F1-Score + ROC-AUC** for any medical classification task.

> 💡 **Rule:** Split worse than 80/20 → worry about imbalance. This 55/45 split is as clean as it gets in real-world medical data.

## 📊 Step 5: Univariate Analysis
> **Univariate = Analyzing ONE variable at a time**
> Check the shape, spread, and outliers of each feature

In [ ]:
# Encode target for correlation analysis
df['target_num'] = (df['Heart Disease'] == 'Presence').astype(int)

numeric_cols     = ['Age', 'BP', 'Cholesterol', 'Max HR', 'ST depression']
categorical_cols = ['Sex', 'Chest pain type', 'FBS over 120', 'EKG results',
                    'Exercise angina', 'Slope of ST', 'Number of vessels fluro', 'Thallium']

print('✅ Numeric columns    :', numeric_cols)
print('✅ Categorical columns:', categorical_cols)

In [ ]:
# Histograms for Numeric Features
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col], bins=40, color='steelblue', edgecolor='black', alpha=0.8)
    axes[i].axvline(df[col].mean(),   color='red',   linestyle='--', linewidth=2, label=f'Mean: {df[col].mean():.1f}')
    axes[i].axvline(df[col].median(), color='green', linestyle='-',  linewidth=2, label=f'Median: {df[col].median():.1f}')
    axes[i].set_title(f'Distribution of {col}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')
    axes[i].legend(fontsize=9)

axes[5].axis('off')
plt.suptitle('📊 Numeric Features Distribution', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('numeric_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Red dashed = Mean | Green solid = Median')
print('💡 If Mean ≠ Median → Distribution is SKEWED')

📊 Numeric Features Distribution — Key Takeaways

1. **Age & Cholesterol are nearly symmetric** — Mean ≈ Median (54.1≈54.0 and 245≈243), bell-shaped distributions. These are the cleanest features, minimal preprocessing needed.

2. **BP shows multimodal peaks** (spikes at 120, 130, 140) — likely due to doctors rounding to nearest 10. Normal behavior in real medical data, not a data error.

3. **Max HR is left-skewed** — Mean (152.8) < Median (157.0), meaning a tail of patients with dangerously low max heart rates pulling the average down. Those low-HR patients are likely your disease cases.

4. **ST depression is heavily right-skewed** — Median is just 0.1 but Mean is 0.7, with a long tail reaching 6+. Most patients have near-zero depression, but disease patients spike it high. Apply **log1p()** before modeling.

> 💡 **Beginner Rule:** Mean ≈ Median → symmetric, safe to use raw. Mean >> Median → right skew → apply `log1p()`. Mean << Median → left skew → consider `square` transformation.

In [ ]:
# Box Plots - Find Outliers!
fig, axes = plt.subplots(1, 5, figsize=(20, 6))

for i, col in enumerate(numeric_cols):
    axes[i].boxplot(df[col], patch_artist=True,
                    boxprops=dict(facecolor='lightblue', color='navy'),
                    medianprops=dict(color='red', linewidth=2),
                    flierprops=dict(marker='o', markerfacecolor='orange', markersize=2))
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_ylabel('Value')

plt.suptitle('📦 Box Plots - Outlier Detection', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Dots outside the whiskers = OUTLIERS')
print('💡 Box = Middle 50% of data (IQR)')

📦 Box Plots — Key Takeaways

1. **Age has almost no outliers** — tight, clean distribution. Most patients are 45–60, a few rare cases below 35. Safest feature, use as-is.

2. **BP & Cholesterol have notable upper outliers** — BP reaching ~200, Cholesterol spiking ~560+. These are medically possible so **don't delete them** — they likely belong to severe disease cases. Use **Winsorization**.

3. **ST depression has the most dense outlier cloud** — the box sits near zero but dots scatter all the way to 6+. Confirms the heavy right-skew. Apply `log1p()` transformation before modeling.

4. **Max HR has the widest IQR (box size)** — high natural variability. Also has very low outliers (~75 bpm) which are clinically significant — those patients' hearts couldn't accelerate, strong disease signal.

> 💡 **Beginner Rule:** Outliers in medical data = **don't blindly remove them.** Use **Winsorization** instead — clip extreme values to the 1st and 99th percentile rather than deleting rows.

In [ ]:
# Categorical Features - Bar Charts
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

cat_labels = {
    'Sex'                    : {0: 'Female', 1: 'Male'},
    'Chest pain type'        : {1: 'Typical Angina', 2: 'Atypical', 3: 'Non-anginal', 4: 'Asymptomatic'},
    'FBS over 120'           : {0: 'No', 1: 'Yes'},
    'EKG results'            : {0: 'Normal', 1: 'ST-T Abnorm', 2: 'LV Hypertrophy'},
    'Exercise angina'        : {0: 'No', 1: 'Yes'},
    'Slope of ST'            : {1: 'Upsloping', 2: 'Flat', 3: 'Downsloping'},
    'Number of vessels fluro': {0: '0', 1: '1', 2: '2', 3: '3'},
    'Thallium'               : {3: 'Normal', 6: 'Fixed Defect', 7: 'Reversible'}
}

for i, col in enumerate(categorical_cols):
    val_counts = df[col].value_counts().sort_index()
    labels = [cat_labels[col].get(k, str(k)) for k in val_counts.index]
    axes[i].bar(labels, val_counts.values, color=sns.color_palette('Set2', len(val_counts)),
                edgecolor='black', linewidth=0.5)
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_ylabel('Count')
    axes[i].tick_params(axis='x', rotation=20)
    for j, v in enumerate(val_counts.values):
        axes[i].text(j, v + 200, f'{v:,}', ha='center', fontsize=8)

plt.suptitle('📊 Categorical Features Distribution', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('categorical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

📊 Categorical Features Distribution — Key Takeaways

1. **Males dominate at 71.5%** (450,283 vs 179,717) — dataset is gender-imbalanced. Model may be slightly biased toward male patterns, keep this in mind during evaluation.

2. **Asymptomatic chest pain (329,179) is the largest group** — over 52% of patients reported no chest pain yet many have disease. Heart disease is often silent and dangerous.

3. **FBS over 120 is heavily skewed** — 91.9% patients have normal blood sugar. This class imbalance makes FBS a **weak standalone predictor** — low information gain for most models.

4. **EKG results: ST-T Abnormality is nearly absent** (only 1,322 cases) — this category is so rare it's almost noise. Consider **merging it with LV Hypertrophy** (category 2) before modeling.

5. **Reversible Thallium defect (246,748) is surprisingly high** — nearly 40% of patients — a strong disease signal worth noting.

> 💡 **Beginner Rule:** If one category has < 1% of total rows (like ST-T Abnorm with 0.2%), consider merging or dropping it — rare categories confuse models more than they help.

## 🔗 Step 6: Bivariate Analysis
> **Bivariate = Analyzing TWO variables together**
> How does each feature relate to Heart Disease?

In [ ]:
# Numeric Features vs Heart Disease - Box Plots
fig, axes = plt.subplots(1, 5, figsize=(22, 6))

for i, col in enumerate(numeric_cols):
    df.boxplot(column=col, by='Heart Disease', ax=axes[i],
               patch_artist=True,
               boxprops=dict(linewidth=1.2))
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Heart Disease')

plt.suptitle('📦 Numeric Features vs Heart Disease', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('bivariate_numeric.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 If boxes are at different heights → Feature is USEFUL for prediction!')

📦 Numeric Features vs Heart Disease — Key Takeaways

1. **Age & Cholesterol boxes barely differ** between Absence/Presence — small median shift only. Weak individual predictors, but still valuable in combination with other features.

2. **BP shows almost identical boxes** for both classes — surprisingly poor separator on its own. Don't over-rely on it as a standalone feature despite its medical reputation.

3. **Max HR is the clearest separator** — Absence group sits noticeably higher (median ~168) vs Presence (median ~150). Lower max HR = stronger disease signal. **Best numeric predictor of the five.**

4. **ST depression tells the strongest story** — Absence median is near 0, Presence median jumps to ~1.5 with a much wider spread.

> 💡 **Beginner Rule:** When two box plots have **clearly different median lines** → feature is a strong predictor. When boxes **heavily overlap** → feature alone won't help much.

In [ ]:
# Categorical Features vs Heart Disease - Stacked Bar
fig, axes = plt.subplots(2, 4, figsize=(22, 12))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    ct = pd.crosstab(df[col], df['Heart Disease'], normalize='index') * 100
    ct.plot(kind='bar', stacked=True, ax=axes[i],
            color=['#2ECC71', '#E74C3C'], edgecolor='black', linewidth=0.5)
    axes[i].set_title(f'{col}\nvs Heart Disease (%)', fontsize=10, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Percentage (%)')
    axes[i].tick_params(axis='x', rotation=30)
    axes[i].legend(title='Disease', fontsize=8)
    axes[i].axhline(y=50, color='gray', linestyle='--', alpha=0.5)

plt.suptitle('📊 Categorical Features vs Heart Disease (Stacked %)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('bivariate_categorical.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 More RED = Higher disease risk for that category')

📊 Categorical Features vs Heart Disease — Key Takeaways

1. **Sex is a strong separator** — Females (0) have ~17% disease rate, Males (1) spike to ~55%. Being male nearly triples the risk in this dataset.

2. **Chest pain type 4 (Asymptomatic) is the most dangerous** — ~70% disease rate despite no pain reported. Type 1 (Typical Angina) is paradoxically the safest. Classic silent killer pattern confirmed.

3. **Exercise angina is the cleanest binary separator** — No exercise angina (0) → ~30% disease. Yes (1) → ~80% disease. The single most decisive categorical feature visually.

4. **Number of vessels fluro shows perfect disease progression** — 0 vessels blocked → ~30% disease, 1 → ~70%, 2 → ~88%, 3 → ~90%+. A textbook monotonic relationship — more blockage = more disease, every single step.

5. **Thallium Reversible defect (7) → ~70% disease rate** vs Normal (3) → ~20%.

> 💡 **Beginner Rule:** The more the red/green ratio **flips dramatically** across categories, the more predictive that feature is. Exercise angina and Vessels fluro are your **star categorical features**.

In [ ]:
# Age Distribution by Heart Disease
fig = px.histogram(df, x='Age', color='Heart Disease',
                   barmode='overlay', nbins=40,
                   color_discrete_map={'Presence': '#E74C3C', 'Absence': '#2ECC71'},
                   title='🎂 Age Distribution by Heart Disease Status',
                   opacity=0.7)
fig.update_layout(xaxis_title='Age', yaxis_title='Count')
fig.show()
print('💡 Overlapping area shows similar age groups — differences show risk patterns')

🎂 Age Distribution by Heart Disease — Key Takeaways

1. **Under 50 → almost entirely green** — disease cases are rare below age 50, healthy patients dominate completely. Young age is a strong protective factor.

2. **The crossover happens at ~55** — red starts competing with green seriously from age 55 onwards. This is the critical risk threshold in this dataset.

3. **Age 58–62 is the danger zone** — red overtakes green, disease cases become the majority.

> 💡 **Beginner Rule:** Where green dominates = model should predict Absence. Where red overtakes = model should lean toward Presence. Tree-based models like XGBoost will automatically learn to split at ~55–60.

In [ ]:
# Scatter Plot: Age vs Max HR colored by Heart Disease (sampled 50K)
sample = df.sample(50000, random_state=42)

fig = px.scatter(sample, x='Age', y='Max HR',
                 color='Heart Disease',
                 color_discrete_map={'Presence': '#E74C3C', 'Absence': '#2ECC71'},
                 title='💓 Age vs Max Heart Rate (sampled 50K)',
                 opacity=0.5,
                 hover_data=['BP', 'Cholesterol'])
fig.show()
print('💡 Patients with lower Max HR tend to have more Heart Disease')

💓 Age vs Max Heart Rate — Key Takeaways

1. **Green dominates the top, red dominates the bottom** — healthy patients consistently achieve higher Max HR, disease patients cluster in the lower half. A clear horizontal separation exists around **Max HR ~150**.

2. **The bottom floor (Max HR < 100) is almost entirely red** — patients who couldn't push their heart above 100 bpm during exercise are nearly all disease cases. A very reliable danger zone.

> 💡 **Beginner Rule:** This 2D pattern is exactly what `df['Age_MaxHR_ratio'] = df['Age'] / df['Max HR']` captures as an engineered feature — we build this in the modeling section!

## 🌡️ Step 7: Correlation Analysis
> Correlation tells us HOW STRONGLY two numeric variables move together
> Range: -1 (opposite) to +1 (same direction), 0 = no relationship

In [ ]:
# Correlation Matrix Heatmap
corr_cols   = numeric_cols + categorical_cols + ['target_num']
corr_matrix = df[corr_cols].corr().round(2)

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            linewidths=0.5, ax=ax, annot_kws={'size': 9})

ax.set_title('🌡️ Correlation Matrix Heatmap', fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Last row (target_num) shows which features correlate most with Heart Disease')

🌡️ Correlation Matrix Heatmap — Key Takeaways

1. **Thallium (0.61) is the single strongest predictor** of heart disease — by far the darkest green in the target row.

2. **Top 5 features by target correlation:** Thallium `0.61`, Chest pain type `0.46`, Exercise angina `0.44`, Vessels fluro `0.44`, Slope of ST `0.42`

3. **Max HR is the only negative correlator (−0.44)** — the orange cell stands out. Higher Max HR = lower disease risk.

4. **BP, Cholesterol, FBS are near-zero with target** — weak individual predictors despite medical reputation.

5. **No multicollinearity problem** — no two features exceed 0.50 with each other. All 13 features carry unique information — keep all of them!

> 💡 **Beginner Rule:** Correlation with target > 0.3 = definitely keep. Near 0 = weak alone but keep for ensemble models. > 0.8 between two features = multicollinearity, drop one.

In [ ]:
# Feature Correlation with Target (Sorted Bar Chart)
target_corr = df[corr_cols].corr()['target_num'].drop('target_num').sort_values()
bar_colors  = ['#E74C3C' if v > 0 else '#2ECC71' for v in target_corr.values]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(target_corr.index, target_corr.values, color=bar_colors, edgecolor='black', linewidth=0.5)
ax.axvline(x=0, color='black', linewidth=1)
ax.set_title('🎯 Feature Correlation with Heart Disease', fontsize=14, fontweight='bold')
ax.set_xlabel('Correlation Coefficient')

for bar, val in zip(bars, target_corr.values):
    ax.text(val + (0.005 if val >= 0 else -0.005), bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', ha='left' if val >= 0 else 'right', fontsize=9)

plt.tight_layout()
plt.savefig('feature_correlation.png', dpi=300, bbox_inches='tight')
plt.show()
print('💡 RED bars = Positive correlation | GREEN bars = Negative correlation')
print('💡 MAGNITUDE (distance from 0) = how strong the feature is')

🎯 Feature Correlation — Key Takeaways

1. **Clear 3-tier feature ranking:**
   - 🔥 **Strong (>0.40):** Thallium, Chest pain type, Exercise angina, Vessels fluro, ST depression, Slope of ST, Max HR — your core model features
   - ⚡ **Moderate (0.20–0.40):** Sex, EKG results, Age — supportive but not decisive
   - 🔵 **Weak (<0.10):** Cholesterol, FBS, BP — least individually useful

2. **Max HR is the lone green bar (−0.44)** — equal strength to the top positive predictors but works in reverse. Never drop a feature just because it's negative.

3. **BP (−0.01) is practically useless alone** — a good reminder that **medical importance ≠ ML importance**.

## 🧪 Step 8: Statistical Tests
> **Why?** Visualizations show patterns, but statistical tests PROVE them!
> T-test for numeric features | Chi-Square for categorical features

In [ ]:
# T-Test: Are means significantly different between Presence/Absence groups?
print('='*65)
print('🧪 T-TEST: Numeric Features vs Heart Disease')
print('='*65)
print(f"{'Feature':<25} {'T-Stat':>10} {'P-Value':>12} {'Significant?':>14}")
print('-'*65)

presence = df[df['Heart Disease'] == 'Presence']
absence  = df[df['Heart Disease'] == 'Absence']

for col in numeric_cols:
    t_stat, p_val = stats.ttest_ind(presence[col], absence[col])
    sig = '✅ YES' if p_val < 0.05 else '❌ NO'
    print(f'{col:<25} {t_stat:>10.3f} {p_val:>12.4f} {sig:>14}')

print('\n💡 P-Value < 0.05 = Statistically SIGNIFICANT difference!')
print('💡 This means the feature IS useful for predicting Heart Disease')

In [ ]:
# Chi-Square Test: Categorical Features vs Heart Disease
print('='*70)
print('🧪 CHI-SQUARE TEST: Categorical Features vs Heart Disease')
print('='*70)
print(f"{'Feature':<28} {'Chi-Stat':>10} {'P-Value':>12} {'Significant?':>14}")
print('-'*70)

for col in categorical_cols:
    ct = pd.crosstab(df[col], df['Heart Disease'])
    chi2, p_val, dof, expected = stats.chi2_contingency(ct)
    sig = '✅ YES' if p_val < 0.05 else '❌ NO'
    print(f'{col:<28} {chi2:>10.1f} {p_val:>12.4f} {sig:>14}')

print('\n💡 P-Value < 0.05 = Strong relationship with Heart Disease confirmed!')

## 📈 Step 9: Advanced Plots

In [ ]:
# Violin Plots - Shows distribution shape + separation
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
key_features = ['Age', 'Max HR', 'ST depression']

for i, col in enumerate(key_features):
    sns.violinplot(data=df, x='Heart Disease', y=col, ax=axes[i],
                   palette={'Presence': '#E74C3C', 'Absence': '#2ECC71'},
                   inner='box')
    axes[i].set_title(f'{col} by Heart Disease', fontsize=12, fontweight='bold')

plt.suptitle('🎻 Violin Plots - Distribution Shape by Disease Status', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('violin_plots.png', dpi=300, bbox_inches='tight')
plt.show()
print('💡 Wide = many data points at that value | Narrow = fewer data points')
print('💡 Fat at DIFFERENT heights = strong predictor | Fat at SAME heights = weak predictor')

🎻 Violin Plots — Key Takeaways

1. **Age violins heavily overlap** — both groups peak around 55–60, confirming age alone is a weak separator.

2. **Max HR violins shift clearly downward for Presence** — Absence is fatter/wider at 160–170, Presence bulges lower at 130–160. Clean separation confirms it's your strongest numeric predictor.

3. **ST depression is the most dramatic contrast** — Absence violin is almost entirely flat near 0, while Presence violin is wide and tall from 0–4, with a long tail to 6+. Two completely different distributions.

> 💡 **Beginner Rule:** Violin width = how many patients have that value. **Fat at different heights = great predictor.** ST depression and Max HR show opposite bulge positions — exactly what a model needs to draw a decision boundary.

In [ ]:
# Heart Disease Rate by Age Group
df['Age Group'] = pd.cut(df['Age'], bins=[29, 39, 49, 59, 69, 80],
                          labels=['30-39', '40-49', '50-59', '60-69', '70+'])

age_disease = df.groupby('Age Group', observed=True)['target_num'].mean() * 100

fig = px.bar(x=age_disease.index.astype(str), y=age_disease.values,
             title='📊 Heart Disease Rate by Age Group (%)',
             labels={'x': 'Age Group', 'y': 'Disease Rate (%)'},
             color=age_disease.values,
             color_continuous_scale='RdYlGn_r',
             text=age_disease.round(1).astype(str) + '%')
fig.update_traces(textposition='outside')
fig.show()
print('💡 Does disease risk increase consistently with age? Check here!')

## 📋 Step 10: EDA Summary

In [ ]:
print('='*60)
print('📋 EDA COMPLETE — KEY FINDINGS')
print('='*60)
print(f'''
📦 DATASET
   Rows           : {df.shape[0]:,}
   Missing values : 0  |  Duplicates: 0  |  Clean ✅
   Target balance : 55.2% Absence / 44.8% Presence

🏆 FEATURE POWER RANKING (by |correlation| with target)
   👑 Thallium              : +0.61  (strongest predictor)
   🔥 Chest pain type       : +0.46
   🔥 Exercise angina       : +0.44
   🔥 Vessels fluro         : +0.44
   🔥 Max HR                : -0.44  (only negative predictor!)
   🔥 ST depression         : +0.43
   🔥 Slope of ST           : +0.42
   ⚡  Sex                   : +0.34
   ⚡  EKG results           : +0.22
   ⚡  Age                   : +0.21
   🔵 Cholesterol           : +0.08
   🔵 FBS over 120          : +0.03
   🔵 BP                    : -0.01  (near zero!)

⚙️  PREPROCESSING PLAN (applied in Step 11)
   ✅ log1p() on ST depression (right-skewed)
   ✅ Winsorize BP & Cholesterol (outliers)
   ✅ Merge EKG category 1 → 2 (only 0.2% of data)
   ✅ Engineer Age/MaxHR ratio feature
   ✅ Scale numeric features (StandardScaler)
   ✅ Drop id column
''')
print('='*60)
print('🚀 EDA Complete! Moving to Model Training...')

---
# 🟢 PART 2 — Model Training, Evaluation & Submission

> 💡 **What changes from EDA to Modeling?**
>
> In EDA we used `df` (a modified copy for analysis). Now we restart fresh
> from `train_raw` and `test_raw` — the untouched original files — to avoid
> any EDA modifications accidentally leaking into the model.

## 🔧 Step 11: Advanced Preprocessing
> Process **train and test together** so identical transformations apply to both

In [ ]:
# ── Restart from clean raw copies ─────────────────────────────────────────
train_model = train_raw.copy()
test_model  = test_raw.copy()

# ── Save ids for submission ────────────────────────────────────────────────
train_ids = train_model['id'].copy()
test_ids  = test_model['id'].copy()

# ── Encode Target ──────────────────────────────────────────────────────────
# Convert 'Presence'/'Absence' → 1/0 (models need numbers, not strings)
train_model['Heart Disease'] = train_model['Heart Disease'].map({'Presence': 1, 'Absence': 0})
print('✅ Target encoded:', train_model['Heart Disease'].value_counts().to_dict())

# ── Separate target + drop id ──────────────────────────────────────────────
y = train_model['Heart Disease'].copy()
train_model.drop(['id', 'Heart Disease'], axis=1, inplace=True)
test_model.drop(['id'], axis=1, inplace=True)

print(f'\n✅ Train features shape : {train_model.shape}')
print(f'✅ Test  features shape : {test_model.shape}')
print(f'✅ Target (y) shape     : {y.shape}')

In [ ]:
# ── Combine train + test for consistent transforms ─────────────────────────
# WHY? So scaler/transforms see same data. Fit on train only, transform both.
train_model['is_train'] = 1
test_model['is_train']  = 0
full = pd.concat([train_model, test_model], axis=0).reset_index(drop=True)

print(f'✅ Combined shape: {full.shape}')
print(f'\n📊 Missing values in combined: {full.isnull().sum().sum()}')

# ── Fix ST depression skew ─────────────────────────────────────────────────
# log1p(x) = log(1+x) — handles zeros safely: log1p(0) = 0
full['ST depression'] = np.log1p(full['ST depression'])
print('✅ ST depression log1p-transformed')

# ── Merge EKG rare category ────────────────────────────────────────────────
# Category 1 has only 0.2% of data → merge into category 2 to reduce noise
full['EKG results'] = full['EKG results'].replace(1, 2)
print('✅ EKG category 1 merged into 2')

# ── Winsorize BP and Cholesterol ───────────────────────────────────────────
# Clips extreme values to 1st/99th percentile — keeps outliers but controls their influence
for col in ['BP', 'Cholesterol']:
    full[col] = mstats.winsorize(full[col], limits=[0.01, 0.01])
    print(f'✅ {col} winsorized (1st–99th percentile)')

## ⚙️ Step 12: Feature Engineering
> Creating new features from existing ones — often the biggest single performance boost
> EDA told us WHAT to engineer — now we build it!

In [ ]:
# ── Feature 1: Age / Max HR ratio ──────────────────────────────────────────
# WHY? EDA scatter showed: old age + low Max HR = strongest disease combination
# A 60-year-old with Max HR 120 is far more concerning than a 30-year-old with 120
full['Age_MaxHR_ratio'] = full['Age'] / (full['Max HR'] + 1)  # +1 avoids division by zero
print('✅ Feature created: Age_MaxHR_ratio')

# ── Feature 2: Age Group (binned) ──────────────────────────────────────────
# WHY? EDA showed clear risk threshold at ~55. Binning captures this non-linearity.
full['Age_Group'] = pd.cut(full['Age'],
                            bins=[0, 40, 50, 60, 70, 100],
                            labels=[0, 1, 2, 3, 4]).astype(int)
print('✅ Feature created: Age_Group')

# ── Feature 3: BP × Cholesterol interaction ────────────────────────────────
# WHY? High BP AND high Cholesterol together is far more dangerous than either alone
full['BP_Chol_interaction'] = full['BP'] * full['Cholesterol'] / 10000  # /10000 to scale down
print('✅ Feature created: BP_Chol_interaction')

# ── Feature 4: Stress Score ────────────────────────────────────────────────
# Combines ST depression + Exercise angina — both are stress-test features
full['Stress_Score'] = full['ST depression'] + full['Exercise angina']
print('✅ Feature created: Stress_Score')

# ── Feature 5: Max HR deficit from expected ────────────────────────────────
# Expected Max HR = 220 - Age (healthy adult formula)
# Deficit = how far below expected the patient is → positive = heart is struggling
full['Expected_MaxHR'] = 220 - full['Age']
full['MaxHR_deficit']  = full['Expected_MaxHR'] - full['Max HR']
print('✅ Feature created: MaxHR_deficit')

print(f'\n📊 Total features after engineering: {full.shape[1] - 1}')  # -1 for is_train

In [ ]:
# ── Scale Numeric Features ─────────────────────────────────────────────────
# CRITICAL RULE: fit scaler on TRAIN only → transform BOTH train and test
# If you fit on combined → data leakage (model sees test statistics during training)

numeric_scale_cols = ['Age', 'BP', 'Cholesterol', 'Max HR', 'ST depression',
                      'Age_MaxHR_ratio', 'BP_Chol_interaction',
                      'Stress_Score', 'Expected_MaxHR', 'MaxHR_deficit']

scaler     = StandardScaler()
train_mask = full['is_train'] == 1

full.loc[train_mask,  numeric_scale_cols] = scaler.fit_transform(full.loc[train_mask,  numeric_scale_cols])
full.loc[~train_mask, numeric_scale_cols] = scaler.transform(    full.loc[~train_mask, numeric_scale_cols])

print('✅ Numeric features scaled (StandardScaler fit on train only — no leakage!)')

# ── Split back into train and test ─────────────────────────────────────────
X      = full[full['is_train'] == 1].drop('is_train', axis=1).reset_index(drop=True)
X_test = full[full['is_train'] == 0].drop('is_train', axis=1).reset_index(drop=True)

print(f'\n✅ X      shape : {X.shape}   (train features)')
print(f'✅ X_test shape : {X_test.shape}   (test features)')
print(f'✅ y      shape : {y.shape}   (target)')
print(f'\n📋 Final feature list:')
print(list(X.columns))

## 🤖 Step 13: Baseline XGBoost
> Always train a **simple baseline model first** — this gives you a reference score to beat.
> If tuning doesn't beat the baseline, your tuning has a bug!

In [ ]:
# ── Train / Validation Split ───────────────────────────────────────────────
# stratify=y ensures same Presence/Absence ratio in both train and val splits
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f'✅ Train size      : {X_train.shape[0]:,} rows')
print(f'✅ Validation size : {X_val.shape[0]:,} rows')
print(f'✅ Stratified split — class ratio preserved in both splits')

In [ ]:
# ── Baseline XGBoost with early stopping ──────────────────────────────────
# early_stopping_rounds=30 → stop if no improvement for 30 rounds (saves time)
baseline_model = XGBClassifier(
    n_estimators        = 500,
    learning_rate       = 0.1,
    max_depth           = 6,
    random_state        = RANDOM_STATE,
    eval_metric         = 'auc',
    early_stopping_rounds = 30,
    verbosity           = 0
)

baseline_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

# Evaluate — use predict_proba for ROC-AUC (needs probabilities, not 0/1!)
val_preds_baseline = baseline_model.predict_proba(X_val)[:, 1]
baseline_auc = roc_auc_score(y_val, val_preds_baseline)

print('='*50)
print('📊 BASELINE MODEL RESULTS')
print('='*50)
print(f'Validation ROC-AUC  : {baseline_auc:.5f}')
print(f'Best iteration      : {baseline_model.best_iteration}')
print('\n💡 This is our reference — Optuna tuning must beat this!')
print('💡 predict_proba()[:,1] gives PROBABILITIES (needed for ROC-AUC)')
print('💡 predict() gives 0/1 LABELS (only for accuracy/F1, NOT for ROC-AUC!)')

## 🔬 Step 14: Hyperparameter Tuning with Optuna

> ### Why Optuna over GridSearch?
> ```
> GridSearchCV   → tries ALL combinations   → dumb brute force, very slow
> RandomSearchCV → tries random combos      → faster but completely blind
> Optuna         → learns from past trials  → SMARTEST & FASTEST ✅
> ```
> Optuna uses **Bayesian optimization** — after each trial it asks "which region
> of the parameter space worked best?" and searches there next. 50 Optuna trials
> beats 500 random trials in practice.

In [ ]:
def objective(trial):
    """
    This function is called once per Optuna trial.
    Each trial tries a different combination of hyperparameters.
    Optuna learns which combinations score best and focuses there.
    """
    params = {
        # n_estimators: number of trees. More trees = more powerful but slower.
        'n_estimators'      : trial.suggest_int(  'n_estimators',       200, 1000),
        # max_depth: how deep each tree goes. Deeper = more complex patterns.
        'max_depth'         : trial.suggest_int(  'max_depth',            3,   10),
        # learning_rate: step size. Small = slower but more precise learning.
        'learning_rate'     : trial.suggest_float('learning_rate',      0.01,  0.3, log=True),
        # subsample: % of rows used per tree. < 1.0 reduces overfitting.
        'subsample'         : trial.suggest_float('subsample',           0.5,  1.0),
        # colsample_bytree: % of features used per tree. Prevents feature dominance.
        'colsample_bytree'  : trial.suggest_float('colsample_bytree',    0.5,  1.0),
        # min_child_weight: min samples needed to split a leaf. Higher = simpler trees.
        'min_child_weight'  : trial.suggest_int(  'min_child_weight',     1,   10),
        # reg_alpha/lambda: L1/L2 regularization. Higher = simpler model.
        'reg_alpha'         : trial.suggest_float('reg_alpha',          1e-5, 10.0, log=True),
        'reg_lambda'        : trial.suggest_float('reg_lambda',         1e-5, 10.0, log=True),
        # gamma: minimum gain to make a split. Higher = more conservative tree.
        'gamma'             : trial.suggest_float('gamma',               0.0,  1.0),
        'random_state'      : RANDOM_STATE,
        'eval_metric'       : 'auc',
        'verbosity'         : 0
    }

    # 5-Fold Stratified CV for reliable score
    # WHY CV instead of single val split?
    # Single split: score depends on which rows ended up in val (luck!)
    # 5-Fold CV: each row is tested once → much more reliable estimate
    cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    model  = XGBClassifier(**params)
    scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)

    return scores.mean()  # Optuna maximizes this value

print('✅ Objective function defined — ready to run Optuna!')

In [ ]:
# ── Run Optuna Study ───────────────────────────────────────────────────────
# n_trials=50 → try 50 combinations → good balance of speed vs quality on Kaggle
study = optuna.create_study(
    direction  = 'maximize',         # We want HIGHER ROC-AUC
    study_name = 'xgb_heart_disease'
)

study.optimize(objective, n_trials=50, show_progress_bar=True)

print('\n' + '='*52)
print('🏆 OPTUNA TUNING RESULTS')
print('='*52)
print(f'Baseline AUC   : {baseline_auc:.5f}')
print(f'Best Tuned AUC : {study.best_value:.5f}')
print(f'Improvement    : +{study.best_value - baseline_auc:.5f}')
print(f'\n🔑 Best Parameters Found:')
for k, v in study.best_params.items():
    print(f'   {k:<25} : {v}')

In [ ]:
# Visualize Optuna Progress
trial_values = [t.value for t in study.trials]
best_so_far  = [max(trial_values[:i+1]) for i in range(len(trial_values))]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# All trial scores
sc = axes[0].scatter(range(len(trial_values)), trial_values,
                     c=trial_values, cmap='RdYlGn', alpha=0.7, s=40)
plt.colorbar(sc, ax=axes[0])
axes[0].axhline(y=baseline_auc,     color='red',   linestyle='--', lw=2, label=f'Baseline: {baseline_auc:.4f}')
axes[0].axhline(y=study.best_value, color='green', linestyle='--', lw=2, label=f'Best: {study.best_value:.4f}')
axes[0].set_xlabel('Trial Number')
axes[0].set_ylabel('ROC-AUC')
axes[0].set_title('All Optuna Trials', fontweight='bold')
axes[0].legend()

# Best score over time
axes[1].plot(best_so_far, color='#2ECC71', linewidth=2.5)
axes[1].fill_between(range(len(best_so_far)), best_so_far, alpha=0.2, color='#2ECC71')
axes[1].set_xlabel('Trial Number')
axes[1].set_ylabel('Best ROC-AUC So Far')
axes[1].set_title('Score Improvement Over Trials', fontweight='bold')

plt.suptitle('🔬 Optuna Hyperparameter Tuning Progress', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('optuna_progress.png', dpi=150, bbox_inches='tight')
plt.show()

## 🏆 Step 15: Final Model Training
> Train on the **FULL training data** (all 630K rows) using the best parameters.
> No holdout split here — we use all available data to maximize model quality.
> We already validated performance with 5-Fold CV in Optuna.

In [ ]:
# ── Build final model with best params ────────────────────────────────────
best_params = study.best_params.copy()
best_params.update({'random_state': RANDOM_STATE, 'verbosity': 0, 'eval_metric': 'auc'})

final_model = XGBClassifier(**best_params)
final_model.fit(X, y)   # ← ALL training data, no holdout

print('✅ Final model trained on complete training data!')
print(f'   Training rows : {X.shape[0]:,}')
print(f'   Features used : {X.shape[1]}')

In [ ]:
# ── Cross-Validation Score of Final Model ─────────────────────────────────
# This is the most honest estimate of what your Kaggle score will be
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(final_model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)

print('='*50)
print('📊 FINAL MODEL — 5-FOLD CV SCORES')
print('='*50)
for i, score in enumerate(cv_scores):
    bar = '█' * int((score - 0.80) * 200)  # Visual bar scaled to score
    print(f'  Fold {i+1}: {score:.5f}  {bar}')
print(f'\n  Mean AUC  : {cv_scores.mean():.5f}')
print(f'  Std       : {cv_scores.std():.5f}  ← lower = more stable/consistent')
print(f'\n💡 Expected Kaggle Public Score : ~{cv_scores.mean():.4f}')
print(f'💡 CV score is MORE reliable than Public LB — always optimize for CV!')

## 📊 Step 16: Full Evaluation
> Deep-dive into model quality — understand WHERE the model is right and wrong

In [ ]:
# ── Retrain on 80% split for honest val evaluation ────────────────────────
eval_model = XGBClassifier(**best_params)
eval_model.fit(X_train, y_train)

val_proba  = eval_model.predict_proba(X_val)[:, 1]   # PROBABILITIES for ROC-AUC
val_labels = (val_proba >= 0.5).astype(int)           # Binary labels at threshold 0.5

print('='*55)
print('📊 COMPLETE EVALUATION REPORT')
print('='*55)
print(f'ROC-AUC  Score : {roc_auc_score(y_val, val_proba):.5f}  ← Primary metric (Kaggle)')
print(f'Accuracy       : {accuracy_score(y_val, val_labels):.5f}  ← Overall % correct')
print(f'F1 Score       : {f1_score(y_val, val_labels):.5f}  ← Balance of precision/recall')
print(f'\n📋 Classification Report:')
print(classification_report(y_val, val_labels, target_names=['Absence (0)', 'Presence (1)']))

In [ ]:
# ROC Curve + Confusion Matrix + Feature Importance — All in one plot
fig, axes = plt.subplots(1, 3, figsize=(21, 6))

# ── Plot 1: ROC Curve ──────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_val, val_proba)
roc_val = auc(fpr, tpr)
axes[0].plot(fpr, tpr, color='#1565c0', lw=2.5, label=f'Tuned XGBoost (AUC = {roc_val:.4f})')
axes[0].plot([0,1],[0,1], color='gray', linestyle='--', lw=1.5, label='Random Classifier (AUC = 0.50)')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='#1565c0')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve', fontweight='bold', fontsize=13)
axes[0].legend(loc='lower right')
# 💡 Area under the curve = ROC-AUC score. Perfect model = 1.0. Random = 0.5.
# The more the curve bulges toward the top-left corner, the better.

# ── Plot 2: Confusion Matrix ───────────────────────────────────────────────
cm = confusion_matrix(y_val, val_labels)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Absence', 'Presence'],
            yticklabels=['Absence', 'Presence'],
            ax=axes[1], linewidths=0.5)
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('True Label')
axes[1].set_title('Confusion Matrix', fontweight='bold', fontsize=13)
# 💡 Reading the confusion matrix:
#   Top-left     → True Negatives  (correctly said Absence)
#   Bottom-right → True Positives  (correctly said Presence)
#   Top-right    → False Positives (said Presence, actually Absence)
#   Bottom-left  → False Negatives (said Absence, actually Presence) ← WORST in medicine!

# ── Plot 3: Feature Importance ────────────────────────────────────────────
feat_imp = pd.Series(
    eval_model.feature_importances_,
    index=X.columns
).sort_values(ascending=True).tail(15)

colors_imp = ['#1565c0' if 'Age_MaxHR' in name or 'Stress' in name or
              'deficit' in name or 'interaction' in name or 'Group' in name
              else 'steelblue' for name in feat_imp.index]

feat_imp.plot(kind='barh', ax=axes[2], color=colors_imp, edgecolor='black', linewidth=0.5)
axes[2].set_title('Top 15 Feature Importances\n(darker = engineered feature)', fontweight='bold', fontsize=12)
axes[2].set_xlabel('Importance Score (XGBoost gain)')
# 💡 Compare this to EDA correlation chart — they should roughly agree!
# If an engineered feature appears high here, our feature engineering worked!

plt.suptitle('📊 Complete Model Evaluation', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Prediction Distribution — does model output look reasonable?
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of predicted probabilities
axes[0].hist(val_proba[y_val == 0], bins=50, alpha=0.6, color='#2ECC71', label='Actual: Absence (0)')
axes[0].hist(val_proba[y_val == 1], bins=50, alpha=0.6, color='#E74C3C', label='Actual: Presence (1)')
axes[0].axvline(x=0.5, color='black', linestyle='--', lw=2, label='Decision threshold = 0.5')
axes[0].set_xlabel('Predicted Probability of Disease')
axes[0].set_ylabel('Count')
axes[0].set_title('Prediction Distribution by True Label', fontweight='bold')
axes[0].legend()
# 💡 Good model: green peaks near 0, red peaks near 1, with minimal overlap

# Calibration check
from sklearn.calibration import calibration_curve
prob_true, prob_pred = calibration_curve(y_val, val_proba, n_bins=10)
axes[1].plot(prob_pred, prob_true, marker='o', color='#1565c0', lw=2, label='Model')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1.5, label='Perfect calibration')
axes[1].set_xlabel('Mean Predicted Probability')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Calibration Curve', fontweight='bold')
axes[1].legend()
# 💡 Good calibration: model line stays close to diagonal.
# If model says "70% chance of disease", ~70% of those patients should actually have it.

plt.suptitle('🔍 Prediction Quality Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('prediction_quality.png', dpi=150, bbox_inches='tight')
plt.show()

## 📤 Step 17: Generate Submission File
> The final step — create the exact CSV format Kaggle expects

> ⚠️ **Most common beginner mistake:** Submitting 0/1 labels instead of probabilities.
> ROC-AUC **requires** probabilities between 0.0 and 1.0. Always use `predict_proba()[:,1]`!

In [ ]:
# ── Generate test predictions ──────────────────────────────────────────────
# Use the FINAL model (trained on all data) for best predictions
# predict_proba()[:, 1] → probability of class 1 (Presence)
test_proba = final_model.predict_proba(X_test)[:, 1]

print(f'✅ Test predictions generated!')
print(f'   Count         : {len(test_proba):,}')
print(f'   Range         : {test_proba.min():.4f} – {test_proba.max():.4f}')
print(f'   Mean prob     : {test_proba.mean():.4f}')
print(f'\n💡 Values are between 0 and 1 — these are PROBABILITIES, not labels. ✅')

In [ ]:
# ── Build submission DataFrame ─────────────────────────────────────────────
submission = pd.DataFrame({
    'id'           : test_ids,
    'Heart Disease': test_proba
})

# ── Run 3 sanity checks before saving ─────────────────────────────────────
print('🔍 Running submission sanity checks...')
assert submission.shape[0] == test_ids.shape[0],          '❌ FAIL: Row count mismatch!'
assert submission['Heart Disease'].between(0, 1).all(),    '❌ FAIL: Probabilities out of 0-1 range!'
assert list(submission.columns) == list(sub.columns),      '❌ FAIL: Column names mismatch with sample_submission!'
print('✅ Check 1 PASSED: Row count matches test.csv')
print('✅ Check 2 PASSED: All probabilities are between 0.0 and 1.0')
print('✅ Check 3 PASSED: Column names exactly match sample_submission.csv')

# ── Save CSV ───────────────────────────────────────────────────────────────
# index=False is CRITICAL — without it, pandas adds an unwanted row number column
submission.to_csv('submission.csv', index=False)
print('\n✅ submission.csv saved successfully!')
print('\n📋 Preview (first 10 rows):')
print(submission.head(10).to_string(index=False))

In [ ]:
# Final Summary
print('='*60)
print('🏆 COMPLETE NOTEBOOK SUMMARY')
print('='*60)
print(f'''
📦 DATA
   Train    : {X.shape[0]:,} rows × {X.shape[1]} features
   Test     : {X_test.shape[0]:,} rows

⚙️  PREPROCESSING
   ✅ Target encoded        (Presence=1, Absence=0)
   ✅ ST depression         log1p() transformed (fixed skew)
   ✅ EKG rare category     merged (1→2, was 0.2% of data)
   ✅ BP & Cholesterol      winsorized (1st–99th percentile)
   ✅ 5 features engineered (ratio, bins, interaction, stress, deficit)
   ✅ Numeric features      StandardScaler (fit on train only — no leakage!)

🔬 TUNING
   Method   : Optuna (50 smart trials, 5-fold Stratified CV)
   Baseline : {baseline_auc:.5f}
   Best CV  : {study.best_value:.5f}
   Gain     : +{study.best_value - baseline_auc:.5f}

📊 FINAL CV SCORE
   Mean AUC : {cv_scores.mean():.5f}
   Std      : ±{cv_scores.std():.5f}

📤 SUBMISSION
   File     : submission.csv
   Rows     : {len(submission):,}
   Format   : id + probability (0.0 to 1.0)
   Checks   : 3/3 passed ✅
''')
print('='*60)
print('🚀 Ready to submit to Kaggle!')

---
## 🗺️ How to Submit — Complete Beginner Guide

### Method 1: From Kaggle Notebook (Easiest ✅)
```
1. Run All Cells  →  Kernel menu → Run All
2. Click 'Save Version' button (top right corner)
3. Select 'Save & Run All (Commit)'
4. Wait ~10 minutes for the notebook to finish running
5. After commit completes → click 'Submit to Competition' button
   (this appears automatically when output files are detected)
6. Select 'submission.csv' from output files
7. Click Submit → wait 1–2 minutes for your score to appear
```

### Method 2: Download & Upload Manually
```
1. Run all cells → submission.csv appears in the Output tab
2. Click Output tab → Download submission.csv to your computer
3. Go to: kaggle.com/competitions/playground-series-s6e2
4. Click 'Submit Predictions' button
5. Upload your submission.csv file
6. Add a description (optional) → click Submit
7. Score appears on Leaderboard within 1–2 minutes
```

### ⚠️ 5 Mistakes That Kill Your Submission
```
❌ MISTAKE 1: Using predict() instead of predict_proba()[:, 1]
   → ROC-AUC REQUIRES probabilities 0.0–1.0, not 0 or 1!
   → Fix: test_proba = final_model.predict_proba(X_test)[:, 1]

❌ MISTAKE 2: Wrong column names
   → Must EXACTLY match sample_submission.csv (case-sensitive!)
   → Fix: print(sub.columns) and copy them exactly

❌ MISTAKE 3: Wrong number of rows
   → Must match test.csv row count exactly (no more, no less)
   → Fix: assert submission.shape[0] == len(test_ids)

❌ MISTAKE 4: Forgetting index=False when saving
   → Without it pandas adds an extra row number column → Kaggle rejects it
   → Fix: submission.to_csv('submission.csv', index=False)

❌ MISTAKE 5: Using wrong ids
   → Must use test_ids from test.csv, NOT sequential 0,1,2,3...
   → Fix: submission['id'] = test_ids  (saved at the very start)
```

### 📈 Understanding Your ROC-AUC Score
```
0.50 → Random model (coin flip)     — terrible, check your code
0.70 → Decent baseline              — you're learning, keep going!
0.80 → Good model                   — solid preprocessing + features
0.85 → Very good                    — well-tuned model
0.90 → Excellent                    — top tier for this dataset
0.95+→ Suspicious → check for data leakage!

Public vs Private Leaderboard:
  Public  → Score during competition (30% of test data)
  Private → Final score after deadline (70% of test data)

💡 ALWAYS optimize for CV score, NOT Public LB score!
   CV is based on all training data → more reliable.
   Public LB can fluctuate due to small sample → don't overfit to it.
```